# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NameRectified/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections in order - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

## 1. Method choice and why

Which method from the toolkit, and why it fits your lane.

This lane ranks pages by CTR and engagement opportunity. The output is a sorted queue for a reviewer. The toolkit offers Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting for this type of problem.

I chose Logistic Regression first because it gives readable coefficients. In a lane where the goal is decision-support, knowing which signals drive the score is as important as the score itself. A coefficient for impressions, CTR gap, or engagement rate tells a reviewer something they can act on.

I chose Random Forest second because it can capture interactions that LR cannot, like whether the CTR gap matters more for some content types than others. If RF beats LR by a wide margin, those interactions exist and are worth describing. If RF ties LR, the simpler model is the right choice.

Both models output probabilities. I rank pages by predicted probability of having a persistent CTR gap. The label is persistent_gap: 1 if the page was below its position-tier median CTR in both the feature window (Jan-Feb 2026) and the label window (March 2026).

In [1]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

data = con.sql(f"""
    SELECT f.content_hash_id,
           MAX(f.client_hash_id) AS client_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           c.content_type,
           c.main_intent,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clicks_label
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Loaded {len(data):,} pages with complete data')

def assign_tier(pos):
    if pos <= 3:
        return 'top_3'
    if pos <= 10:
        return 'page_1'
    if pos <= 20:
        return 'striking'
    if pos <= 50:
        return 'page_3_5'
    return 'deep'

data['ctr_fw'] = data['clicks_fw'] / data['impressions_fw'] * 100
data['engagement_rate_fw'] = data['engaged_sessions_fw'] / data['sessions_fw'] * 100
data['position_tier'] = data['avg_pos_fw'].apply(assign_tier)

tier_med = data.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100
)
data['tier_median_ctr'] = data['position_tier'].map(tier_med)
data['tier_ctr_gap'] = data['tier_median_ctr'] - data['ctr_fw']

data['ctr_label'] = data['clicks_label'] / data['impressions_label'] * 100
data['gap_label'] = data['tier_median_ctr'] - data['ctr_label']

data['persistent_gap'] = ((data['tier_ctr_gap'] > 0.1) & (data['gap_label'] > 0.1)).astype(int)

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')
data = data.fillna(0)

data['log_impressions_fw'] = np.log1p(data['impressions_fw'])
data['log_sessions_fw'] = np.log1p(data['sessions_fw'])

unique_clients = data['client_hash_id'].nunique()
print(f'Unique clients in data: {unique_clients}')
print(f'Class balance: {data["persistent_gap"].mean():.1%} positive (persistent_gap=1)')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 120,258 pages with complete data
Unique clients in data: 47
Class balance: 49.3% positive (persistent_gap=1)


/var/folders/yy/t95v8nfd5mg826zr_b6pmwf00000gp/T/ipykernel_88916/1835362196.py:60: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tier_med = data.groupby('position_tier', observed=True).apply(


## 2. Split design

Grouped by client? Time-aware? Say why this split is honest for your question.

I use a client-holdout split. 80% of clients go into training, 20% are held out for testing. Pages from the same client often share content patterns and search profiles. If I trained on some pages from a client and tested on others, the model could memorize client-level quirks instead of learning generalizable signals. Holding out entire clients shows whether the model works for new clients the system has never seen.

The split is not time-aware because the decision point is fixed: March 1, 2026. All pages share the same feature window (Jan-Feb) and label window (March). There is no rolling time dimension within this window. Time-aware splits would matter if the data covered multiple decision points.

Random seed is fixed at 42 so the split is reproducible.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

clients = data['client_hash_id'].unique()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(data, groups=data['client_hash_id']))

train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

train_clients = train['client_hash_id'].nunique()
test_clients = test['client_hash_id'].nunique()
print(f'Train: {len(train):,} pages from {train_clients} clients')
print(f'Test:  {len(test):,} pages from {test_clients} clients')
print(f'Test base rate: {test["persistent_gap"].mean():.1%}')

Train: 112,968 pages from 37 clients
Test:  7,290 pages from 10 clients
Test base rate: 13.3%


## 3. Train + compare vs my baseline

Same data, same metric, same split as your Week-4 baseline. Show the table.

The baseline from Week 4 is: has_volume times max(tier_ctr_gap, 0) times impressions_fw. I recompute it here on the test set using the same formula. That way the comparison uses the same split.

I train Logistic Regression and Random Forest on the same training set and evaluate all three on the same test set. The metric is precision@K for K=10 and K=50, which matches the Week-4 evaluation.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

num_features = ['log_impressions_fw', 'ctr_fw', 'avg_pos_fw', 'pos_volatility_fw',
                'engagement_rate_fw', 'log_sessions_fw', 'tier_ctr_gap']
cat_features = ['content_type', 'main_intent', 'position_tier']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

X_train = preprocessor.fit_transform(train[num_features + cat_features])
X_test = preprocessor.transform(test[num_features + cat_features])
y_train = train['persistent_gap']
y_test = test['persistent_gap']

lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

has_volume = (test['impressions_fw'] >= 500).astype(int)
ctr_gap = test['tier_ctr_gap'].clip(lower=0)
bl_score = has_volume * ctr_gap * test['impressions_fw']

def precision_at_k(score, y, k):
    top = score.nlargest(k).index if len(score) >= k else score.nlargest(len(score)).index
    return y.loc[top].mean()

bl_p10 = precision_at_k(pd.Series(bl_score.values, index=test.index), y_test, 10)
bl_p50 = precision_at_k(pd.Series(bl_score.values, index=test.index), y_test, 50)
lr_p10 = precision_at_k(pd.Series(lr_probs, index=test.index), y_test, 10)
lr_p50 = precision_at_k(pd.Series(lr_probs, index=test.index), y_test, 50)
rf_p10 = precision_at_k(pd.Series(rf_probs, index=test.index), y_test, 10)
rf_p50 = precision_at_k(pd.Series(rf_probs, index=test.index), y_test, 50)

print(f'{"Method":<25} {"Precision@10":<15} {"Precision@50":<15}')
print('-' * 55)
print(f'{"Baseline (gap x volume)":<25} {bl_p10:<15.1%} {bl_p50:<15.1%}')
print(f'{"Logistic Regression":<25} {lr_p10:<15.1%} {lr_p50:<15.1%}')
print(f'{"Random Forest":<25} {rf_p10:<15.1%} {rf_p50:<15.1%}')
print(f'{"Test base rate":<25} {y_test.mean():<15.1%}')

Method                    Precision@10    Precision@50   
-------------------------------------------------------
Baseline (gap x volume)   30.0%           44.0%          
Logistic Regression       10.0%           10.0%          
Random Forest             0.0%            10.0%          
Test base rate            13.3%          


## 4. Errors and interpretation

Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.

In [4]:
feature_names = num_features + list(
    preprocessor.named_transformers_['cat']
    .get_feature_names_out(cat_features)
)
lr_coefs = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

rf_imp = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
rf_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_imp.importances_mean
}).sort_values('importance', ascending=False)

print('Top 5 LR coefficients (absolute):')
print(lr_coefs.head(5).to_string(index=False))
print()
print('Top 5 RF permutation importances:')
print(rf_imp_df.head(5).to_string(index=False))
print()

test_scores = pd.DataFrame({
    'bl_score': bl_score.values,
    'lr_prob': lr_probs,
    'rf_prob': rf_probs,
    'y_true': y_test,
    'impressions_fw': test['impressions_fw'].values,
    'tier_ctr_gap': test['tier_ctr_gap'].values,
    'position_tier': test['position_tier'].values,
    'content_type': test['content_type'].values,
    'main_intent': test['main_intent'].values
}, index=test.index)

test_scores['bl_pred'] = (test_scores['bl_score'] > test_scores['bl_score'].median()).astype(int)
test_scores['lr_pred'] = (test_scores['lr_prob'] > 0.5).astype(int)
test_scores['rf_pred'] = (test_scores['rf_prob'] > 0.5).astype(int)

print('Error breakdown:')
fp = test_scores[(test_scores['lr_pred'] == 1) & (test_scores['y_true'] == 0)]
fn = test_scores[(test_scores['lr_pred'] == 0) & (test_scores['y_true'] == 1)]
print(f'  False positives (model says yes, true is no): {len(fp)}')
print(f'  False negatives (model says no, true is yes): {len(fn)}')
print()

print('3 wrong cases (false positives with highest LR probability):')
top_fp = fp.sort_values('lr_prob', ascending=False).head(3)
for idx, row in top_fp.iterrows():
    print(f'  impressions={row["impressions_fw"]:.0f}, gap={row["tier_ctr_gap"]:.4f}, '
          f'tier={row["position_tier"]}, intent={row["main_intent"]}')
    print(f'  Model gave them high probability but the gap did not persist in March.')
    print()

print('3 wrong cases (false negatives with highest true gap):')
top_fn = fn.sort_values('tier_ctr_gap', ascending=False).head(3)
for idx, row in top_fn.iterrows():
    print(f'  impressions={row["impressions_fw"]:.0f}, gap={row["tier_ctr_gap"]:.4f}, '
          f'tier={row["position_tier"]}, intent={row["main_intent"]}')
    print(f'  Large gap existed but model predicted no persistence. Unusual pattern.')
    print()

print('Summary:')
print('The baseline beats both models on held-out clients. The test clients have a much')
print('lower persistent gap rate (13.3% vs 49.3% in training), which is a hard shift for')
print('a model trained on the training clients.')
print()
print('The simple rule (gap times volume) generalizes better because it does not learn')
print('client-specific patterns. It relies on two transparent conditions: does the page')
print('have enough volume, and is its CTR below the tier median. Those conditions hold')
print('even for new clients with different base rates.')
print()
print('The models overfit to training client patterns. LR and RF both use tier_ctr_gap')
print('as a feature but also learn content type, main intent, position tier, and')
print('engagement signals that do not transfer to the test clients. For this label,')
print('a transparent rule is more reliable than a learned model for new clients.')

Top 5 LR coefficients (absolute):
                    feature  coefficient
               tier_ctr_gap    11.595250
         position_tier_deep    -7.415421
                     ctr_fw    -6.655758
        position_tier_top_3     2.465836
content_type_feedly article    -2.437724

Top 5 RF permutation importances:
                  feature  importance
       log_impressions_fw    0.003169
        pos_volatility_fw    0.001742
main_intent_transactional    0.000988
main_intent_informational    0.000864
   position_tier_striking    0.000631

Error breakdown:
  False positives (model says yes, true is no): 3030
  False negatives (model says no, true is yes): 76

3 wrong cases (false positives with highest LR probability):
  impressions=785, gap=0.4059, tier=top_3, intent=transactional
  Model gave them high probability but the gap did not persist in March.

  impressions=844, gap=0.4059, tier=top_3, intent=informational
  Model gave them high probability but the gap did not persist in March

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ - then submit your repo URL on the card. Done.